# REINVENT4 Mol2Mol

This notebook evaluates REINVENT4 Mol2Mol using the same fixed 1,000 parent compounds used by the other benchmark methods.

Before sampling, all 1,000 parent SMILES are checked against the vocabulary of the Mol2Mol prior.

- Supported parents are passed to REINVENT4.
- Unsupported parents are retained as benchmark failures and saved separately.
- The original 1,000-parent set is not replaced or resampled.

This allows the benchmark to report both practical applicability to the full parent set and generation performance among parents that REINVENT4 can accept.


## Preparation


### Import library


In [ ]:
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")

sys.path.append("..")

from configs.benchmark_config import (
    N_CANDIDATES_PER_PARENT,
    RANDOM_SEED,
)
from utils.descriptors import calc_descriptors
from utils.parents import prepare_parent
from utils.records import append_candidate_record

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path("..").resolve()

INPUT_SMI = PROJECT_ROOT / "data/reinvent4_mol2mol_input.smi"
CONFIG_FILE = PROJECT_ROOT / "configs/reinvent4_mol2mol_sampling.toml"
RAW_OUTPUT_FILE = PROJECT_ROOT / "results/reinvent4_mol2mol_raw.csv"
OUTPUT_FILE = PROJECT_ROOT / "results/reinvent4_mol2mol.csv"
UNSUPPORTED_FILE = PROJECT_ROOT / "results/reinvent4_unsupported_parents.csv"
LOG_FILE = PROJECT_ROOT / "results/reinvent4_mol2mol.log"

### Prepare the Mol2Mol prior


In [ ]:
PRIOR_FILE = PROJECT_ROOT / "external/reinvent4_priors/mol2mol_medium_similarity.prior"
PRIOR_URL = "https://zenodo.org/records/15641297/files/mol2mol_medium_similarity.prior?download=1"

PRIOR_FILE.parent.mkdir(parents=True, exist_ok=True)

if not PRIOR_FILE.exists():
    subprocess.run(
        ["curl", "-L", "--fail", PRIOR_URL, "-o", str(PRIOR_FILE)],
        check=True,
    )

### Import parent compounds


In [ ]:
df = pd.read_csv(PROJECT_ROOT / "data/chembl_1000_parents.csv")
print(f"Number of parent compounds: {len(df)}")
df.head()

### Check Mol2Mol vocabulary compatibility

REINVENT4 validates every input SMILES against the vocabulary stored in the selected Mol2Mol prior. The same REINVENT4 vocabulary utilities are used here before the full sampling run.


In [ ]:
import importlib.util
from pathlib import Path

import torch
import reinvent

from reinvent.utils import get_tokens_from_vocabulary
from reinvent.utils.config_parse import find_invalid_tokens


def load_reinvent_adapter(prior_file):
    path = (
        Path(reinvent.__file__).parent
        / "runmodes"
        / "create_adapter.py"
    )

    spec = importlib.util.spec_from_file_location(
        "_create_adapter",
        path,
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    return module.create_adapter(
        str(prior_file),
        "inference",
        torch.device("cpu"),
    )


adapter, _, model_type = load_reinvent_adapter(PRIOR_FILE)

if model_type != "Mol2Mol":
    raise ValueError(f"Unexpected model type: {model_type}")

allowed_tokens = get_tokens_from_vocabulary(
    adapter.vocabulary
)[0]


def is_reinvent_supported(smiles):
    return not find_invalid_tokens(
        smiles,
        allowed_tokens,
    )


supported_mask = df["smiles"].apply(
    is_reinvent_supported
)

df_supported = df[supported_mask].copy()
df_unsupported = df[~supported_mask].copy()

print(f"Supported: {len(df_supported)} / {len(df)}")

### Save unsupported parents


In [ ]:
UNSUPPORTED_FILE.parent.mkdir(parents=True, exist_ok=True)
df_unsupported.to_csv(UNSUPPORTED_FILE, index=False)
print(f"Saved: {UNSUPPORTED_FILE}")

### Create REINVENT4 Mol2Mol input

Only vocabulary-compatible parents are passed to REINVENT4 so that one unsupported molecule does not abort sampling for all other parents. Unsupported parents are not replaced by other molecules.


In [ ]:
INPUT_SMI.parent.mkdir(parents=True, exist_ok=True)
with open(INPUT_SMI, "w") as f:
    for smi in df_supported["smiles"]:
        f.write(f"{smi}\n")

with open(INPUT_SMI) as f:
    input_smiles = [line.strip() for line in f if line.strip()]

assert input_smiles == df_supported["smiles"].tolist()
print(f"Saved: {INPUT_SMI}")
print(f"Number of supported input SMILES: {len(input_smiles)}")

## Compounds generation: REINVENT4 Mol2Mol


In [ ]:
CONFIG_FILE.parent.mkdir(parents=True, exist_ok=True)
RAW_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

toml_text = f'''
run_type = "sampling"
device = "cpu"

[parameters]
model_file = "{PRIOR_FILE}"
smiles_file = "{INPUT_SMI}"
sample_strategy = "multinomial"
temperature = 1.0
output_file = "{RAW_OUTPUT_FILE}"
num_smiles = {N_CANDIDATES_PER_PARENT}
unique_molecules = false
randomize_smiles = false
'''
CONFIG_FILE.write_text(toml_text.strip() + "\n")
print(CONFIG_FILE.read_text())

In [ ]:
import os

env = os.environ.copy()
env["KMP_DUPLICATE_LIB_OK"] = "TRUE"

command = [
    "reinvent",
    "-l",
    str(LOG_FILE),
    "--seed",
    str(RANDOM_SEED),
    str(CONFIG_FILE),
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True,
    env=env,
)

if result.returncode != 0:
    if LOG_FILE.exists():
        print(LOG_FILE.read_text(errors="replace")[-5000:])

    print(result.stdout[-2000:])
    print(result.stderr[-5000:])

    raise RuntimeError(
        f"REINVENT4 failed with return code {result.returncode}."
    )

if not RAW_OUTPUT_FILE.exists():
    raise RuntimeError(
        "REINVENT4 finished without creating the expected raw output file."
    )

REINVENT4 completed.
Raw output: /Users/tesak/github/scaffold-tuner-experiments/results/reinvent4_mol2mol_raw.csv


### Calculate descriptor changes


In [ ]:
raw = pd.read_csv(RAW_OUTPUT_FILE)

parent_lookup = {}

for _, row in df_supported.iterrows():
    mol = Chem.MolFromSmiles(row["smiles"])
    if mol is None:
        continue

    smiles = Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=True,
    )

    parent_lookup[smiles] = row["chembl_id"]


records = []
seen = set()

for _, row in raw.iterrows():
    if pd.isna(row["Input_SMILES"]) or pd.isna(row["SMILES"]):
        continue

    parent = Chem.MolFromSmiles(row["Input_SMILES"])
    generated = Chem.MolFromSmiles(row["SMILES"])

    if parent is None or generated is None:
        continue

    parent_smiles = Chem.MolToSmiles(
        parent,
        canonical=True,
        isomericSmiles=True,
    )

    generated_smiles = Chem.MolToSmiles(
        generated,
        canonical=True,
        isomericSmiles=True,
    )

    chembl_id = parent_lookup.get(parent_smiles)
    if chembl_id is None:
        continue

    if generated_smiles == parent_smiles:
        continue

    key = (chembl_id, generated_smiles)
    if key in seen:
        continue

    seen.add(key)

    parent_desc = calc_descriptors(parent)

    append_candidate_record(
        records=records,
        chembl_id=chembl_id,
        parent_smiles=parent_smiles,
        generated_smiles=generated_smiles,
        parent_desc=parent_desc,
    )

df_reinvent = pd.DataFrame(records)

print(df_reinvent.shape)
df_reinvent.head()

Raw rows: 92900
Missing generated SMILES: 2
Missing input SMILES: 0


## 4. Save generated products

The output columns intentionally match the Random substitution output so that all methods can be evaluated with the same downstream analysis code.

In [ ]:
OUTPUT_FILE = "../results/reinvent4_mol2mol.csv"

df_reinvent.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")

(61388, 15)
Saved: /Users/tesak/github/scaffold-tuner-experiments/results/reinvent4_mol2mol.csv


,chembl_id,parent_smiles,generated_smiles,hbd_parent,hbd_generated,delta_hbd,hba_parent,hba_generated,delta_hba,ar_parent,ar_generated,delta_ar,rotb_parent,rotb_generated,delta_rotb
0,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,COc1cc(-c2ccc[nH]2)[nH]c1/C=C1/C=CC(Cc2ccccc2)=N1,2,2,0,4,2,-2,2,3,1,9,5,-4
1,CHEMBL441229,C=C1C(O)C(O)C(O)C(O)C1O,C=C1CC(O)C(O)C(O)C1O,5,4,-1,5,4,-1,0,0,0,0,0,0
2,CHEMBL1068,NC(=O)N1c2ccccc2CC(=O)c2ccccc21,NC(=O)N1c2ccccc2CCc2c1ccc1ccccc21,1,1,0,2,1,-1,2,3,1,0,0,0
3,CHEMBL105373,COc1cc(-n2sc3ncccc3c2=O)cc(OC)c1OC,COc1ccc(-n2sc3ncccc3c2=O)cc1,0,0,0,6,4,-2,3,3,0,4,2,-2
4,CHEMBL313560,O=C(O)CCCc1ccc(NC(=O)c2ccccc2[N+](=O)[O-])cc1,O=C(O)CCc1ccc(NC(=O)c2cccc(Oc3ccccc3)c2)cc1,2,2,0,4,3,-1,2,3,1,7,7,0


## Downstream comparison

`results/reinvent4_unsupported_parents.csv` records parents that REINVENT4 Mol2Mol cannot accept because of vocabulary limitations.

For analysis, report both:

- Full-set applicability/performance: denominator = all 1,000 fixed parents; unsupported REINVENT4 parents count as failures.
- Supported-parent comparison: compare all methods on the subset of parents accepted by REINVENT4.
